# Step 7 — Full Pipeline Integrity Audit ✅

Walks Steps 0 through 6, checking **input count vs output count** at every stage (catching silent drops)
and **known-bug regression checks** — specifically re-testing for the exact failure patterns we already
found and fixed earlier in this project, so a future change can't silently reintroduce them.

Every check prints ✅ PASS, ⚠️ WARN, or ❌ FAIL. Final cell gives one summary count.

**This does not change any pipeline output — read-only audit.**

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found.")

from config import DATA_ROOT, STEP0_DIR, STEP1_DIR, STEP2_DIR, STEP3_DIR, STEP4_DIR, STEP5_DIR, OUTPUT_ROOT
STEP6_DIR = OUTPUT_ROOT / "step_6"

audit_log = []   # (step, check_name, status, detail)

def check(step, name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    icon = "\u2705" if condition else "\u274c"
    print(f"{icon} [{step}] {name}" + (f" — {detail}" if detail else ""))
    audit_log.append((step, name, status, detail))

def warn(step, name, condition, detail=""):
    status = "OK" if condition else "WARN"
    icon = "\u2705" if condition else "\u26a0\ufe0f"
    print(f"{icon} [{step}] {name}" + (f" — {detail}" if detail else ""))
    audit_log.append((step, name, status, detail))

print("Audit initialized.\n")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
Audit initialized.



In [2]:
# ── STEP 0 audit ──────────────────────────────────────────────
print("=== STEP 0 — Dataset Preparation ===")

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

n_samples = len(samples_index)
check("Step0", "Sample count == 404 (nuScenes-mini)", n_samples == 404, f"got {n_samples}")

missing_sensor_meta = [sid for sid, info in samples_index.items()
                        if not (STEP0_DIR / info["folder"] / "sensor_meta.json").exists()]
check("Step0", "Every sample has sensor_meta.json", len(missing_sensor_meta) == 0,
      f"{len(missing_sensor_meta)} missing")

# Regression check: "folder" must be RELATIVE, not an absolute leaked path
sample_folder_values = [info["folder"] for info in samples_index.values()]
has_absolute_path = any((":\\" in f or f.startswith("/")) for f in sample_folder_values)
check("Step0", "REGRESSION: 'folder' field is relative, not an absolute path",
      not has_absolute_path, "found absolute path in index!" if has_absolute_path else "clean")

sensor_counts = []
for sid, info in list(samples_index.items())[:20]:
    with open(STEP0_DIR / info["folder"] / "sensor_meta.json") as f:
        meta = json.load(f)
    sensor_counts.append(len(meta["sensor_channels"]))
check("Step0", "All 12 sensor channels present (sampled 20 files)",
      all(c == 12 for c in sensor_counts), f"counts seen: {set(sensor_counts)}")
print()

=== STEP 0 — Dataset Preparation ===
✅ [Step0] Sample count == 404 (nuScenes-mini) — got 404
✅ [Step0] Every sample has sensor_meta.json — 0 missing
✅ [Step0] REGRESSION: 'folder' field is relative, not an absolute path — clean
✅ [Step0] All 12 sensor channels present (sampled 20 files) — counts seen: {12}



In [3]:
# ── STEP 1.1/1.2/1.3 audit ───────────────────────────────────
print("=== STEP 1.1 — Camera Preprocessing ===")
with open(STEP1_DIR / "camera_meta.json") as f:
    camera_meta = json.load(f)
check("Step1.1", "Camera samples count matches Step 0", len(camera_meta) == n_samples,
      f"{len(camera_meta)} / {n_samples}")
cam_channel_counts = [len(v) for v in camera_meta.values()]
check("Step1.1", "All samples have 6 camera channels", all(c == 6 for c in cam_channel_counts),
      f"counts seen: {set(cam_channel_counts)}")
has_calib = all("calibrated_sensor" in list(v.values())[0] for v in camera_meta.values() if v)
check("Step1.1", "Calibration present per camera entry", has_calib)

print("\n=== STEP 1.2 — Radar Parsing ===")
radar_dir = STEP1_DIR / "radar"
radar_sample_folders = [p for p in radar_dir.iterdir() if p.is_dir()]
check("Step1.2", "Radar sample folders count matches Step 0", len(radar_sample_folders) == n_samples,
      f"{len(radar_sample_folders)} / {n_samples}")
sample_radar_file = next(radar_sample_folders[0].glob("RADAR_*.json"))
with open(sample_radar_file) as f:
    radar_sample_data = json.load(f)
check("Step1.2", "REGRESSION: radar file has dict structure with 'points' + 'calibration'",
      "points" in radar_sample_data and "calibration" in radar_sample_data)

print("\n=== STEP 1.3 — LiDAR Parsing ===")
lidar_dir = STEP1_DIR / "lidar"
lidar_sample_folders = [p for p in lidar_dir.iterdir() if p.is_dir()]
check("Step1.3", "LiDAR sample folders count matches Step 0", len(lidar_sample_folders) == n_samples,
      f"{len(lidar_sample_folders)} / {n_samples}")
sample_lidar = np.load(lidar_sample_folders[0] / "lidar_raw.npy")
check("Step1.3", "lidar_raw.npy has 7 columns (x,y,z,intensity,distance,azimuth,elevation)",
      sample_lidar.shape[1] == 7, f"got shape {sample_lidar.shape}")
with open(lidar_sample_folders[0] / "lidar_meta.json") as f:
    lidar_meta_sample = json.load(f)
check("Step1.3", "lidar_meta.json has ego_pose + calibration",
      "ego_pose" in lidar_meta_sample and "sensor_to_ego_translation" in lidar_meta_sample)
print()

=== STEP 1.1 — Camera Preprocessing ===
✅ [Step1.1] Camera samples count matches Step 0 — 404 / 404
✅ [Step1.1] All samples have 6 camera channels — counts seen: {6}
✅ [Step1.1] Calibration present per camera entry

=== STEP 1.2 — Radar Parsing ===
✅ [Step1.2] Radar sample folders count matches Step 0 — 404 / 404
✅ [Step1.2] REGRESSION: radar file has dict structure with 'points' + 'calibration'

=== STEP 1.3 — LiDAR Parsing ===
✅ [Step1.3] LiDAR sample folders count matches Step 0 — 404 / 404
✅ [Step1.3] lidar_raw.npy has 7 columns (x,y,z,intensity,distance,azimuth,elevation) — got shape (34688, 7)
✅ [Step1.3] lidar_meta.json has ego_pose + calibration



In [4]:
# ── STEP 2.1/2.2/2.3/2.3.1 audit ─────────────────────────────
print("=== STEP 2.1 — LiDAR Ground Removal + Clustering ===")
lidar2_dir = STEP2_DIR / "lidar"
sample_lidar2 = next(p for p in lidar2_dir.iterdir() if p.is_dir())

has_ground = (sample_lidar2 / "lidar_ground.npy").exists()
has_nonground = (sample_lidar2 / "lidar_nonground.npy").exists()
check("Step2.1", "REGRESSION: lidar_ground.npy exists (this was the original missing-file bug)",
      has_ground)
check("Step2.1", "lidar_nonground.npy exists", has_nonground)

if has_nonground and (sample_lidar2 / "lidar_cluster_labels.npy").exists():
    nonground = np.load(sample_lidar2 / "lidar_nonground.npy")
    labels = np.load(sample_lidar2 / "lidar_cluster_labels.npy")
    check("Step2.1", "REGRESSION: cluster_labels length matches nonground points (exact membership, not crude match)",
          len(labels) == len(nonground), f"{len(labels)} labels vs {len(nonground)} points")

print("\n=== STEP 2.2 — Radar Filtering ===")
radar2_dir = STEP2_DIR / "radar"
sample_radar2_folder = next(p for p in radar2_dir.iterdir() if p.is_dir())
sample_radar2_file = next(sample_radar2_folder.glob("RADAR_*.json"))
with open(sample_radar2_file) as f:
    radar2_data = json.load(f)
check("Step2.2", "REGRESSION: filtered radar output preserves dict structure ('points' + 'calibration')",
      "points" in radar2_data and "calibration" in radar2_data)
if radar2_data["points"]:
    has_both_vel = "vx" in radar2_data["points"][0] and "vx_comp" in radar2_data["points"][0]
    check("Step2.2", "REGRESSION: both raw (vx) and compensated (vx_comp) velocity present", has_both_vel)

print("\n=== STEP 2.3 / 2.3.1 — YOLO Detection + Global 3D Projection ===")
yolo_global_dir = STEP2_DIR / "yolo_global"
sample_yg_folder = next(p for p in yolo_global_dir.iterdir() if p.is_dir())
sample_yg_file = next(sample_yg_folder.glob("CAM_*.json"), None)
if sample_yg_file:
    with open(sample_yg_file) as f:
        yg_data = json.load(f)
    if yg_data:
        has_3d_field = "has_3d_position" in yg_data[0]
        check("Step2.3.1", "has_3d_position field present on detections", has_3d_field)
print()

=== STEP 2.1 — LiDAR Ground Removal + Clustering ===
✅ [Step2.1] REGRESSION: lidar_ground.npy exists (this was the original missing-file bug)
✅ [Step2.1] lidar_nonground.npy exists
✅ [Step2.1] REGRESSION: cluster_labels length matches nonground points (exact membership, not crude match) — 20865 labels vs 20865 points

=== STEP 2.2 — Radar Filtering ===
✅ [Step2.2] REGRESSION: filtered radar output preserves dict structure ('points' + 'calibration')
✅ [Step2.2] REGRESSION: both raw (vx) and compensated (vx_comp) velocity present

=== STEP 2.3 / 2.3.1 — YOLO Detection + Global 3D Projection ===
✅ [Step2.3.1] has_3d_position field present on detections



In [5]:
# ── STEP 3.1/3.2/3.3 audit — the most important regression checks ──
print("=== STEP 3.1/3.2/3.3 — Tracking ===")

def audit_tuple_tracks(track_dir, sensor_name):
    files = list(track_dir.glob("track_*.json"))
    duplicate_sample_tracks = 0
    length_2_count = 0
    total = len(files)
    for f in files:
        with open(f) as fh:
            points = json.load(fh)
        sample_ids_in_track = [p[0] for p in points]
        if len(sample_ids_in_track) != len(set(sample_ids_in_track)):
            duplicate_sample_tracks += 1   # REGRESSION signal: double-assignment bug
        if len(points) == 2:
            length_2_count += 1
    return total, duplicate_sample_tracks, length_2_count

for sensor_name, track_dir in [("lidar", STEP3_DIR / "lidar"), ("radar", STEP3_DIR / "radar")]:
    total, dupes, len2 = audit_tuple_tracks(track_dir, sensor_name)
    check(f"Step3.{sensor_name}", f"REGRESSION: no track has the same sample_id twice (double-assignment bug)",
          dupes == 0, f"{dupes}/{total} tracks affected")
    warn(f"Step3.{sensor_name}", "Fraction of minimum-length (2-frame) tracks not dominant",
         len2 / total < 0.5 if total > 0 else True, f"{len2}/{total} ({len2/total*100:.0f}%)" if total > 0 else "n/a")

# Camera uses a different (dict/trajectory) format
cam_track_dir = STEP3_DIR / "camera"
cam_files = list(cam_track_dir.glob("track_*.json"))
cam_dupes = 0
for f in cam_files:
    with open(f) as fh:
        data = json.load(fh)
    sids = [pt["sample_id"] for pt in data["trajectory"]]
    if len(sids) != len(set(sids)):
        cam_dupes += 1
check("Step3.camera", "REGRESSION: no camera track has the same sample_id twice", cam_dupes == 0,
      f"{cam_dupes}/{len(cam_files)} tracks affected")

cross_cam_tracks = sum(
    1 for f in cam_files
    if len({pt["camera"] for pt in json.load(open(f))["trajectory"]}) > 1
)
check("Step3.camera", "Unified tracker produces some cross-camera tracks (design goal)",
      cross_cam_tracks > 0, f"{cross_cam_tracks} tracks span 2+ cameras")
print()

=== STEP 3.1/3.2/3.3 — Tracking ===
✅ [Step3.lidar] REGRESSION: no track has the same sample_id twice (double-assignment bug) — 0/4513 tracks affected
✅ [Step3.lidar] Fraction of minimum-length (2-frame) tracks not dominant — 1350/4513 (30%)
✅ [Step3.radar] REGRESSION: no track has the same sample_id twice (double-assignment bug) — 0/10425 tracks affected
✅ [Step3.radar] Fraction of minimum-length (2-frame) tracks not dominant — 2982/10425 (29%)
✅ [Step3.camera] REGRESSION: no camera track has the same sample_id twice — 0/1326 tracks affected
✅ [Step3.camera] Unified tracker produces some cross-camera tracks (design goal) — 651 tracks span 2+ cameras



In [6]:
# ── STEP 4 audit ──────────────────────────────────────────────
print("=== STEP 4 — Fusion ===")

fused_df = pd.read_csv(STEP4_DIR / "fused_tracks_all.csv")

valid_sensor_tokens = {"lidar", "radar", "camera"}
all_tokens = set()
for s in fused_df["sensors"].unique():
    all_tokens.update(s.split("+"))
check("Step4", "'sensors' column only contains known sensor names",
      all_tokens.issubset(valid_sensor_tokens), f"found: {all_tokens}")

dupe_pairs = fused_df.duplicated(subset=["fused_id", "sample_id"]).sum()
check("Step4", "REGRESSION: no (fused_id, sample_id) pair appears twice (fusion-level double-assignment)",
      dupe_pairs == 0, f"{dupe_pairs} duplicates found")

multi_sensor_pct = fused_df["sensors"].str.contains(r"\+").mean() * 100
print(f"  Info: {multi_sensor_pct:.1f}% of fused points combine 2+ sensors "
      f"(context for the merge-rate discussion — current INTRA_FRAME_MERGE_THRESH)")
print()

=== STEP 4 — Fusion ===
✅ [Step4] 'sensors' column only contains known sensor names — found: {'camera', 'radar', 'lidar'}
✅ [Step4] REGRESSION: no (fused_id, sample_id) pair appears twice (fusion-level double-assignment) — 0 duplicates found
  Info: 12.4% of fused points combine 2+ sensors (context for the merge-rate discussion — current INTRA_FRAME_MERGE_THRESH)



In [7]:
# ── STEP 5 audit ──────────────────────────────────────────────
print("=== STEP 5 — TTC ===")

for name in ["lidar", "radar", "camera", "fused"]:
    ttc_df = pd.read_csv(STEP5_DIR / f"ttc_{name}.csv")
    check(f"Step5.{name}", "REGRESSION: timestamp column present", "timestamp" in ttc_df.columns)

    finite_ttc = ttc_df[np.isfinite(ttc_df["ttc"])]["ttc"]
    if len(finite_ttc) > 0:
        check(f"Step5.{name}", "REGRESSION: no TTC value exceeds MAX_VALID_TTC cap (60s) — outlier bug check",
              finite_ttc.max() <= 60.01, f"max seen: {finite_ttc.max():.1f}s")
        warn(f"Step5.{name}", "Mean and median TTC are reasonably close (no extreme skew)",
             abs(finite_ttc.mean() - finite_ttc.median()) < 5,
             f"mean={finite_ttc.mean():.2f}, median={finite_ttc.median():.2f}")
print()

=== STEP 5 — TTC ===
✅ [Step5.lidar] REGRESSION: timestamp column present
✅ [Step5.lidar] REGRESSION: no TTC value exceeds MAX_VALID_TTC cap (60s) — outlier bug check — max seen: 59.8s
✅ [Step5.lidar] Mean and median TTC are reasonably close (no extreme skew) — mean=10.36, median=5.56
✅ [Step5.radar] REGRESSION: timestamp column present
✅ [Step5.radar] REGRESSION: no TTC value exceeds MAX_VALID_TTC cap (60s) — outlier bug check — max seen: 59.9s
✅ [Step5.radar] Mean and median TTC are reasonably close (no extreme skew) — mean=10.45, median=6.69
✅ [Step5.camera] REGRESSION: timestamp column present
✅ [Step5.camera] REGRESSION: no TTC value exceeds MAX_VALID_TTC cap (60s) — outlier bug check — max seen: 59.7s
✅ [Step5.camera] Mean and median TTC are reasonably close (no extreme skew) — mean=9.04, median=5.47
✅ [Step5.fused] REGRESSION: timestamp column present
✅ [Step5.fused] REGRESSION: no TTC value exceeds MAX_VALID_TTC cap (60s) — outlier bug check — max seen: 59.9s
✅ [Step5.fused] Me

In [8]:
# ── STEP 6 audit ──────────────────────────────────────────────
print("=== STEP 6 — Evaluation Metrics ===")

metrics_path = STEP6_DIR / "evaluation_metrics.csv"
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    check("Step6", "All 4 sensors present in evaluation table",
          set(metrics_df["Sensor"]) == {"lidar", "radar", "camera", "fused"})
    check("Step6", "REGRESSION: no MAE/RMSE in the tens-of-thousands (near-zero closing-speed bug)",
          metrics_df["MAE"].max() < 100, f"max MAE: {metrics_df['MAE'].max():.1f}")
    warn("Step6", "n_matched_pairs is a reasonable fraction of total tracks (locked-identity fix working)",
         (metrics_df["n_matched_pairs"] > 0).all())
else:
    print("  Step 6 evaluation_metrics.csv not found — run Step 6 first.")
print()

=== STEP 6 — Evaluation Metrics ===
✅ [Step6] All 4 sensors present in evaluation table
✅ [Step6] REGRESSION: no MAE/RMSE in the tens-of-thousands (near-zero closing-speed bug) — max MAE: 3.8
✅ [Step6] n_matched_pairs is a reasonable fraction of total tracks (locked-identity fix working)



In [9]:
# ── FINAL SUMMARY ────────────────────────────────────────────
audit_df = pd.DataFrame(audit_log, columns=["step", "check", "status", "detail"])

n_pass = (audit_df["status"] == "PASS").sum()
n_fail = (audit_df["status"] == "FAIL").sum()
n_ok = (audit_df["status"] == "OK").sum()
n_warn = (audit_df["status"] == "WARN").sum()

print("=" * 55)
print(f"AUDIT SUMMARY: {n_pass} PASS, {n_fail} FAIL, {n_ok} OK, {n_warn} WARN")
print("=" * 55)

if n_fail > 0:
    print("\n\u274c FAILED CHECKS (fix these first):")
    display(audit_df[audit_df["status"] == "FAIL"])

if n_warn > 0:
    print("\n\u26a0\ufe0f WARNINGS (worth a look, not necessarily broken):")
    display(audit_df[audit_df["status"] == "WARN"])

audit_path = OUTPUT_ROOT / "step_7_audit_report.csv"
audit_df.to_csv(audit_path, index=False)
print(f"\nFull report saved: {audit_path}")

AUDIT SUMMARY: 34 PASS, 0 FAIL, 7 OK, 0 WARN

Full report saved: F:\Sensor fusion Research\output\step_7_audit_report.csv
